# 1. Cleaning

Reads the files in `data/raw` and writes a tidy CSV for each into
`data/clean/`.
Crime type headings are translated here too. 

In [ ]:
import sys
import pandas as pd
import os

sys.path.append("..")
from src.utils import read_table, to_number, clean_name, clean_crime_type, get_ine_code, as_int

RAW = "../data/raw"
CLEAN = "../data/clean"

os.makedirs(CLEAN, exist_ok=True)

## 1.1 Municipalities

The master table covers the whole of Spain, so it is filtered to province 28, which leaves 179 municipalities.

Three details: 
- The file is semicolon separated and encoded in latin-1
- `COD_INE` has 11 digits because it also identifies entities below the municipality, and
the INE municipality code is the first 5
- `SUPERFICIE` is in hectares
with a comma as the decimal separator

In [ ]:
raw = pd.read_csv(f"{RAW}/master_municipalities.csv", sep=";", encoding="latin-1", dtype=str)

madrid = raw[raw.COD_PROV == "28"]

municipalities = pd.DataFrame({
    "ine_code": madrid.COD_INE.str[:5],
    "municipality_name": madrid.NOMBRE_ACTUAL.apply(clean_name),
    "surface_km2": madrid.SUPERFICIE.apply(to_number) / 100, 
})

municipalities.to_csv(f"{CLEAN}/municipalities.csv", index=False)

municipalities.head()

## 1.2 Population

Two header rows: 
- The first has Total / Hombres / Mujeres
- The second has the years

In [ ]:
(sex_row, year_row), data = read_table(f"{RAW}/population_municipalities.xlsx",  n_header=2)
sex_map = {"Total": "T", "Hombres": "M", "Mujeres": "F"}

rows = []
for _, r in data.iterrows():
    for col in data.columns[1:]:
        year = to_number(year_row[col])
        rows.append({
            "ine_code": get_ine_code(r[0]),
            "year": int(year),
            "sex": sex_map.get(str(sex_row[col]).strip(), "T"),
            "population": to_number(r[col]),
        })

population = pd.DataFrame(rows)
population = population[population["ine_code"].notna()]
population = as_int(population, ["population"])
population.to_csv(f"{CLEAN}/population.csv", index=False)

population.head()

## 1.3 Crimes

One file per year covering the whole of Spain. It sits at a different place in every year, so it is found by
looking for the autonomous community heading rather than by row number.

Until 2023 municipalities are named without their INE code, from 2024 the code is included; `clean_name` handles
both.

In [ ]:
rows = []
for year in range(2019, 2026):
    (crime_row,), data = read_table(f"{RAW}/crimes/crimes_{year}.xlsx", n_header=1)

    labels = data[0].astype(str)
    start = labels[labels.str.contains(r"MADRID \(COMUNIDAD", na=False)].index[0]
    end = labels[labels.str.contains(r"MURCIA \(REGION", na=False)].index[0]

    for i in range(start, end):
        r = data.loc[i]
        for col in data.columns[1:]:
            name = crime_row[col]
            rows.append({
                "year": year,
                "municipality_name": clean_name(r[0]),
                "is_region": i == start, # the first row of the block is the region, the rest municipalities
                "crime_code": clean_crime_type(name, "mun"),
                "crime_count": to_number(r[col]),
            })

crimes = pd.DataFrame(rows)
crimes.to_csv(f"{CLEAN}/crimes.csv", index=False)

crimes.head()

## 1.4 Recorded and cleared offences (whole region)

One row per crime type, one column per year. The two files have exactly the
same shape.

In [ ]:
recorded_rows = []
cleared_rows = []

for rows, filename in [(recorded_rows, "recorded_offences"),(cleared_rows, "cleared_offences")]:
    (year_row,), data = read_table(f"{RAW}/{filename}.xlsx", n_header=1)

    for _, r in data.iterrows():
        for col in data.columns[1:]:
            year = to_number(year_row[col])
            rows.append({
                "year": int(year),
                "crime_code": clean_crime_type(r[0], "reg"),
                "crime_count": to_number(r[col]),
            })

recorded_offences = pd.DataFrame(recorded_rows)
cleared_offences = pd.DataFrame(cleared_rows)

recorded_offences.to_csv(f"{CLEAN}/recorded_offences.csv", index=False)
cleared_offences.to_csv(f"{CLEAN}/cleared_offences.csv", index=False)

print("recorded_offences")
display(recorded_offences.head())
print("cleared_offences")
display(cleared_offences.head())

## 1.5 Victims and offenders by age and sex (whole region)

- `Victims` have seven age groups starting at 0 and three sex columns, the third
being "Se desconoce", genuinely unknown

- `Offenders` have five age groups starting at 14, plus a sixth block "TOTAL edad" which is a total.
Their third sex column is "Ambos sexos", which is also a total

In [ ]:
sex_map = {"Masculino": "M", "Femenino": "F", "Se desconoce": "U", "Ambos sexos": "TOTAL"}
age_map = {
    "0-13 años": "0-13", "14-17 años": "14-17", "18-30 años": "18-30",
    "31-40 años": "31-40", "41-64 años": "41-64", "65 y más años": "65+",
    "Más 64 años": "65+", "Edad desconocida": "U", "TOTAL edad": "TOTAL",
}

# (output name, folder on disk, file name prefix)
sources = [("victims", "victimizations", "victimizations"), 
           ("offenders", "arrested_and_investigated_persons", "arrested_and_investigated_persons")]

for who, folder, prefix in sources:
    rows = []
    for year in range(2019, 2025):
        (age_row, sex_row), data = read_table(f"{RAW}/{folder}/{prefix}_{year}.xlsx", n_header=2)
        for _, r in data.iterrows():
            for col in data.columns[1:]:
                age = age_map.get(str(age_row[col]).strip())
                sex = sex_map.get(str(sex_row[col]).strip())
                rows.append({
                    "year": year, 
                    "crime_code": clean_crime_type(r[0], "reg"),
                    "age": age, 
                    "sex": sex,
                    "count": to_number(r[col]),
                })

    df = pd.DataFrame(rows)
    df = df[(df.age != "TOTAL") & (df.sex != "TOTAL")] 
    df.to_csv(f"{CLEAN}/{who}.csv", index=False)
    
    print(f"\n{who}")
    display(df.head())
    

### 7 files are now in `data/clean/`